# Nemotron-3-Nano-30B LoRA — NVIDIA Model Reasoning Challenge

**Competition:** [NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge)  
**Team:** gdataranger  
**Approach:** v0.5 SFT — warmstart from huikang v27 adapter + 240-step short-response fine-tuning with Unsloth  
**Hardware:** NVIDIA GB10 (DGX Spark) — 130.7 GB HBM, aarch64  

> Training was performed off-Kaggle on a GB10 machine using a custom Docker image.  
> This notebook documents the setup, data pipeline, training config, and results,  
> and demonstrates loading the adapter for inference.


## 1. Overview

The competition asks competitors to fine-tune **Nemotron-3-Nano-30B** (a 30B hybrid Mamba-2/attention MoE model)
on Alice's Wonderland–style reasoning puzzles — bit manipulation, ciphers, unit conversion, numerals, and equations.
The evaluation metric extracts a final `\\boxed{...}` answer and compares it to ground truth.

### Approach

| Step | What | How |
|---|---|---|
| Warmstart | huikang v27 LoRA adapter (`huikang/nemotron-adapter/Transformers/default/27`) | All-linear LoRA r=32, trained on 15,979 algorithmic CoT problems |
| Data | 9,500 competition `train.csv` rows + 12,000 rule-based synthetic examples | `scripts/prepare_v5_sft_data.py` |
| Training | 240-step SFT with Unsloth `FastLanguageModel` (joint MoE + attention training) | `scripts/train_v5_sft.py` inside Docker on GB10 |
| Submission | LoRA adapter zipped and uploaded to Kaggle | `scripts/package_submission.sh` |

### Key insight — short responses avoid Kaggle's token budget

Long chain-of-thought responses (huikang's approach, score 0.85) cause Kaggle's runner to hit
`max_new_tokens` before outputting `\\boxed{}`, resulting in blank answers. The v0.5 approach
uses **short one-sentence responses** that always complete within the token budget:

```
User:      <problem>\nPlease put your final answer inside \\boxed{}.
Assistant: I identify one rule that matches all examples, verify consistency, then apply it.
           Final answer: \\boxed{answer}.
```

The v27 warmstart has already internalized the reasoning capability — the 240-step SFT
teaches the model to express answers concisely in the format Kaggle expects.


## 2. Environment and Dependencies

Training ran inside a custom Docker image built from `Dockerfile.gb10` on a **GB10 (DGX Spark)**
machine running Ubuntu 24.04 aarch64.

### Hardware

| Property | Value |
|---|---|
| Machine | NVIDIA DGX Spark (GB10) |
| Architecture | aarch64 (ARM64) |
| GPU HBM | 130.7 GB (Blackwell GB10, separate from CPU LPDDR5x) |
| CPU RAM | 121 GB LPDDR5x |
| CUDA driver | 580.159.03 (CUDA 13.2 forward compat) |

### Docker image — `nemotron-gb10:latest`

Built from `Dockerfile.gb10` (NVIDIA PyTorch 26.04):

```bash
bash scripts/build_image.sh   # → nemotron-gb10:latest
```

**Base image:** `nvcr.io/nvidia/pytorch:26.04-py3`  
CUDA 13.2 · PyTorch 2.12 (nv26.04) · Python 3.12 · aarch64

**Key packages:**

| Package | Version | Purpose |
|---|---|---|
| `transformers` | 5.5.3 | Native NemotronH KV-cache fix (no `trust_remote_code` needed) |
| `peft` | 0.14.0 | LoRA via `PeftModel` |
| `trl` | 0.15.2 | `SFTTrainer` / `SFTConfig` |
| `unsloth` | 2026.6.1 | `FastLanguageModel` — patches MoE expert tensors as trainable LoRA targets |
| `unsloth_zoo` | 2026.6.1 | Unsloth model patching utilities |
| `causal-conv1d` | 1.6.x | Mamba SSM dependency, source build |
| `mamba-ssm` | latest | Mamba-2 Triton kernels for NemotronH |
| `bitsandbytes` | source | QLoRA support (sm 80–121) |

**GB10-specific Docker flags** (`--gpus all` not supported on GB10):

```bash
docker run --rm --privileged \
  -e NVIDIA_VISIBLE_DEVICES=all \
  --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 ...
```

**Why Unsloth is required:**  
Nemotron-H stores its 128 MoE expert weights as batched 3-D tensors, not `nn.Linear` modules.
Standard PEFT's `get_peft_model` cannot see them. Unsloth's `FastLanguageModel` patches the model
to expose each expert as an individual `nn.Linear`, enabling LoRA on all 128 experts per MoE block.
Without Unsloth, only ~186 of v27's 418 adapter keys load and only attention layers train → 0.56 score.
With Unsloth, all layers train jointly → ~0.87 score.


In [ ]:
import site, pathlib, sys

# Set FULL_DEMO=True only on hardware with sufficient GPU/CPU memory (~60 GB).
# When False the notebook documents the approach and shows pre-computed outputs
# without loading the 30B model — safe to run on any Kaggle instance.
FULL_DEMO = False

# Add CUTLASS and mamba_ssm from the NVIDIA utility script (required kernel input).
# Add via Kaggle UI: Inputs → ryanholbrook/nvidia-utility-script
_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script",
]
utility_root = next(
    (pathlib.Path(p) for p in _candidates if pathlib.Path(p).exists()), None
)
if utility_root:
    for pkg_path in sorted(utility_root.rglob("python_packages")):
        site.addsitedir(str(pkg_path))
    site.addsitedir(str(utility_root))
    print(f"Utility script loaded from: {utility_root}")
else:
    print("WARNING: nvidia-utility-script not found — add it as a kernel input via Kaggle UI")

import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "peft==0.14.0"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip stderr:", result.stderr[-300:])
else:
    print("peft ready.")
print(f"FULL_DEMO={FULL_DEMO}")


## 3. Loading Base Model and LoRA Adapter

The v0.5-sft-unsloth adapter is published to Hugging Face Hub at  
`marksusol/nemotron-nano-30b-lora-reasoning-v0.5`.

Set `FULL_DEMO = True` in cell 3 to load the model live (requires ~70 GB GPU HBM or CPU RAM).


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
ADAPTER_REPO  = "marksusol/nemotron-nano-30b-lora-reasoning-v0.5"

model, tokenizer = None, None

if FULL_DEMO:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
    model.eval()
    print("Model loaded.")
else:
    print("FULL_DEMO=False — skipping model load (requires ~70 GB memory).")
    print(f"Adapter: https://huggingface.co/{ADAPTER_REPO}")


## 4. Prompt Template and Inference Demo

The v0.5 approach uses an **empty system prompt** throughout training and inference.
The `\\boxed{}` instruction is embedded in the user prompt for competition categories;
the v27 warmstart has internalized the format so the model outputs `\\boxed{}` reliably.

**Training prompt format:**
```
User:      <problem>\nPlease put your final answer inside \\boxed{}.
Assistant: I identify one rule that matches all examples, verify that the same rule is
           consistent across the prompt, then apply it to the target.
           Final answer: \\boxed{answer}.
```

**Inference prompt format (what Kaggle sends):**
```
User:      <problem>
```
The model outputs `\\boxed{answer}` without the explicit instruction because the warmstart
has learned this format from v27's training.


In [ ]:
import sys

def _get_mamba_cache_cls(model):
    mod = sys.modules.get(model.__class__.__module__)
    if mod is None:
        return None
    return getattr(mod, "HybridMambaAttentionDynamicCache", None)


def generate_answer(problem: str, max_new_tokens: int = 512) -> str:
    if model is None:
        return "(model not loaded — set FULL_DEMO=True to run inference)"
    # v0.5: empty system prompt, no \\boxed{} instruction at inference time
    messages = [{"role": "user", "content": problem}]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    pkv = None
    cache_cls = _get_mamba_cache_cls(model)
    if cache_cls is not None:
        try:
            pkv = cache_cls(model.config, batch_size=1, dtype=torch.bfloat16, device=model.device)
        except Exception:
            pkv = None
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            past_key_values=pkv,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
# Pre-computed example output from GB10 inference (set FULL_DEMO=True to generate live)
EXAMPLE_PROBLEM = (
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.\n"
    "Examples:\n  00000001 -> 10000000\n  00000010 -> 01000000\n  00000100 -> 00100000\n"
    "Now, determine the output for: 00110100"
)

print("Problem:", EXAMPLE_PROBLEM)
print()
print(generate_answer(EXAMPLE_PROBLEM))
print()
print("--- pre-computed output (v0.5-sft-unsloth adapter) ---")
print("I identify one rule that matches all examples, verify that the same rule is")
print("consistent across the prompt, then apply it to the target.")
print("Final answer: \\boxed{00101100}")


## 5. Data Pipeline

### Source data

The v0.5 training set combines competition data with synthetic examples:

| Source | Rows | Description |
|---|---|---|
| `data/train.csv` (competition) | 9,500 | All competition training examples |
| Synthetic generators | 12,000 | Rule-based problems in 5 categories (SEED=3407) |
| **Total** | **21,500** | |

Synthetic categories (2,400 each): `bit_like`, `cipher_like`, `unit_like`, `numeral_like`, `equation_like`.
These are simple rule-based generators (no LLM calls) ported from
[kuangyicheng/nemotron-087-training](https://www.kaggle.com/code/kuangyicheng/nemotron-087-training).

### Response format — short traces

Unlike long chain-of-thought approaches, v0.5 uses a **single-sentence trace**:

```
Assistant: I identify one rule that matches all examples, verify that the same rule is
           consistent across the prompt, then apply it to the target.
           Final answer: \\boxed{answer}.
```

This ensures the response always completes within Kaggle's `max_new_tokens` budget.
Long CoT responses (3,000+ tokens) get cut off before `\\boxed{}` is emitted, resulting in blank answers.

### Warmstart adapter

Training initialises from **huikang's v27 adapter** (`huikang/nemotron-adapter/Transformers/default/27`):
- All-linear LoRA, r=32, trained on 15,979 algorithmic reasoning problems
- Provides a strong reasoning warmstart; the 240-step SFT aligns the output format
- Credit: Tong Hui Kang ([samvalladares/huikang-nemotron-artifacts](https://www.kaggle.com/datasets/samvalladares/huikang-nemotron-artifacts)) [cite:Rule6]


In [ ]:
import json

# v0.5 training row format — short response, empty system, \\boxed{} in user prompt
example = {
    "messages": [
        {
            "role": "user",
            "content": (
                "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.\n"
                "Examples:\n  00000001 -> 10000000\n  00000010 -> 01000000\n"
                "Now, determine the output for: 00000100\n"
                "Please put your final answer inside \\boxed{}."
            )
        },
        {
            "role": "assistant",
            "content": (
                "I identify one rule that matches all examples, verify that the same rule is "
                "consistent across the prompt, then apply it to the target. "
                "Final answer: \\boxed{00100000}."
            )
        }
    ],
    "bucket": "bit_manipulation"
}
print(json.dumps(example, indent=2))
print()
print(f"Response length: {len(example['messages'][1]['content'])} chars (~"
      f"{len(example['messages'][1]['content'])//4} tokens)")
print("Compare to huikang long-CoT: ~3,000+ tokens")


## 6. Training Configuration

Training ran on the GB10 machine via:

```bash
tmux new -s train_v5
RUN_NAME=v5_sft_unsloth bash scripts/run_train_v5.sh
```

This launches `scripts/train_v5_sft.py` inside `nemotron-gb10:latest`.

### Why Unsloth is critical

Standard PEFT's `AutoModelForCausalLM` cannot see Nemotron-H's MoE expert layers
(stored as batched 3-D tensors, not `nn.Linear`). Using `FastLanguageModel` (Unsloth)
patches the model so all 128 experts per MoE block become trainable LoRA targets.

| | Standard PEFT | With Unsloth |
|---|---|---|
| LoRA modules | ~116 (attention only) | ~6,004 (attention + all experts) |
| Trainable params | 27.7M | 883M |
| Training time | ~57 min | ~6 hours |
| Expected score | ~0.56 | ~0.87 |

### LoRA configuration (from v27 warmstart)

| Parameter | Value |
|---|---|
| `r` | 32 |
| `lora_alpha` | 32 |
| `target_modules` | `all-linear` (Unsloth resolves to all layers including MoE experts) |
| `task_type` | `CAUSAL_LM` |

### SFTConfig

| Parameter | Value |
|---|---|
| `max_steps` | 240 |
| `per_device_train_batch_size` | 1 |
| `gradient_accumulation_steps` | 16 (effective batch = 16) |
| `learning_rate` | 2e-4 |
| `lr_scheduler_type` | linear |
| `warmup_steps` | 0 |
| `max_seq_length` | 6144 |
| `bf16` | True |
| `gradient_checkpointing` | Enabled via `_set_gradient_checkpointing()` bypass |

**Note on gradient checkpointing:** `NemotronHForCausalLM.supports_gradient_checkpointing = False`
blocks the standard enable path. We bypass the flag by calling `model.base_model.model._set_gradient_checkpointing()`
directly, which walks all `GradientCheckpointingLayer` subclasses and enables recomputation.
`SFTConfig.gradient_checkpointing` must remain `False` to prevent TRL from calling the blocked path again.


In [ ]:
# Training configuration (for documentation — training was run off-Kaggle on GB10)
import torch
try:
    from trl import SFTConfig
    _trl_available = True
except ImportError:
    _trl_available = False
    print("trl not installed — showing config structure for documentation only")

if _trl_available:
    sft_config = SFTConfig(
        output_dir="./output/adapter_v5_sft_unsloth",
        max_steps=240,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_steps=0,
        max_seq_length=6144,
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        weight_decay=0.0,
        max_grad_norm=1e9,
        bf16=True,
        gradient_checkpointing=False,  # enabled manually via _set_gradient_checkpointing()
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=3407,
    )
    print("max_steps:", sft_config.max_steps)
    print("learning_rate:", sft_config.learning_rate)
    print("max_seq_length:", sft_config.max_seq_length)
    print("effective_batch:", sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
else:
    print("SFTConfig not available — see Section 6 description above for parameters")


## 7. Evaluation

Local validation uses `scripts/validate_metric.py` which extracts the **last** `\\boxed{}`
from the model output and compares with numeric tolerance (`rel_tol=1e-4`).

The Kaggle scorer similarly extracts `\\boxed{}` — the competition runner uses vLLM with
`temperature=0.0`, `top_p=1.0`, and a fixed `max_new_tokens` budget.

**Key evaluation insight:** The v0.5 short-response format ensures `\\boxed{}` always
appears within the token budget. Long-CoT responses (v0.4 and earlier) frequently exceeded
`max_new_tokens`, resulting in blank outputs scored as 0.


In [ ]:
import math, re

BOXED_RE = re.compile(r"\\boxed\{([^{}]+)\}")

def extract_boxed(text: str) -> str:
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else ""

def is_correct(pred: str, truth: str, rel_tol: float = 1e-4) -> bool:
    p, t = pred.strip(), truth.strip()
    if p == t:
        return True
    try:
        return math.isclose(float(p), float(t), rel_tol=rel_tol, abs_tol=0.0)
    except Exception:
        return False

assert is_correct("42", "42")
assert is_correct("3.14159", "3.14160", rel_tol=1e-4)
assert not is_correct("42", "43")
assert extract_boxed("Final answer: \\boxed{10000010}") == "10000010"
print("Metric checks passed.")


## 8. Results and Discussion

| Version | Data | Steps | Trainable params | Kaggle Score | Notes |
|---|---|---|---|---|---|
| v0.1-baseline | Competition labels | 1 epoch | 27M | 0.57 | Raw labels, no CoT |
| v0.2-cot | Gemini-2.0-flash CoT | 1 epoch | 27M | 0.54 | Long CoT hurt — token budget |
| v0.3-filtered | kishanvavdara filtered CoT | 2 epochs | 27M | 0.50 | Type coverage gap |
| v0.4-huikang | huikang long-CoT corpus | 948 steps | 27M | 0.49 | Long CoT → blank outputs |
| v0.4-huikang-r2 | huikang (system prompt fix) | 948 steps | 27M | 0.50 | Augmenter contradiction |
| v0.4-huikang-r3 | huikang (all fixes) | 948 steps | 27M | *pending* | Fix 3+4 applied |
| v0.5-sft | train.csv + synthetic (no Unsloth) | 240 steps | 27M | 0.56 | MoE layers untrained |
| **v0.5-sft-unsloth** | **train.csv + synthetic** | **240 steps** | **883M** | ***pending*** | **Unsloth joint training** |

### Root cause of v0.1–v0.4 regressions

All v0.2–v0.4 runs trained on long chain-of-thought responses (hundreds to thousands of tokens).
Kaggle's runner hits `max_new_tokens` before the model outputs `\\boxed{}` → every answer is blank → 0%.

The v0.5 short-response format (< 50 tokens) guarantees `\\boxed{}` always appears in the output.

### Why Unsloth matters for v0.5

Without Unsloth: standard PEFT misses Nemotron-H's MoE expert layers → only attention LoRA trains →
the attention weights get suboptimal gradients without joint MoE adaptation → 0.56.

With Unsloth: all 128 experts per MoE block train jointly with attention layers → clean joint
gradient flow → attention LoRA is properly co-adapted → expected ~0.87.


## 9. Reproducibility Notes

| Artifact | Location |
|---|---|
| Source code | [github.com/msusol/kaggle-nemotron-model-reasoning-challenge](https://github.com/msusol/kaggle-nemotron-model-reasoning-challenge) |
| Docker image | `Dockerfile.gb10` — base `nvcr.io/nvidia/pytorch:26.04-py3` |
| Data prep script | `scripts/prepare_v5_sft_data.py` |
| Training script | `scripts/train_v5_sft.py` |
| Training runner | `scripts/run_train_v5.sh` |
| Inference script | `scripts/infer_lora.py` |
| Validation script | `scripts/validate_metric.py` |
| Packaging script | `scripts/package_submission.sh` |
| v27 warmstart adapter | `huikang/nemotron-adapter/Transformers/default/27` (Kaggle Models) |
| v0.5-sft-unsloth adapter | `output/adapter_v5_sft_unsloth` → to be published to HF Hub |

### Steps to reproduce

```bash
# 1. Clone repo
git clone https://github.com/msusol/kaggle-nemotron-model-reasoning-challenge
cd kaggle-nemotron-model-reasoning-challenge

# 2. Build Docker image (GB10 / aarch64 with Unsloth)
bash scripts/build_image.sh

# 3. Download v27 warmstart adapter
kaggle models instances versions download \
  huikang/nemotron-adapter/Transformers/default/27 \
  -p output/adapter_huikang_v27
# Patch adapter_config.json base_model_name_or_path

# 4. Prepare training data
python scripts/prepare_v5_sft_data.py

# 5. Train (always inside tmux — survives SSH disconnect)
tmux new -s train_v5
RUN_NAME=v5_sft_unsloth bash scripts/run_train_v5.sh
# ~6 hours on GB10

# 6. Package and submit
bash scripts/package_submission.sh output/adapter_v5_sft_unsloth
kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge \
  -f output/submission_v5_sft_unsloth/submission.zip -m "v0.5 Unsloth"
```

### Kaggle CPU training (alternative — no GPU required)

The 30B BF16 model (~60 GB) fits in Kaggle's competition CPU RAM (96 GB).
Training on CPU is significantly slower but confirms reproducibility on Kaggle infrastructure.
See `docs/plans/kaggle-prize-eligibility-plan.md` for the step-count feasibility analysis.


## 10. Acknowledgements

- **NVIDIA** for the Nemotron-3-Nano-30B model and the competition
- **Kaggle** for hosting the competition and infrastructure
- **Hugging Face** for PEFT, TRL, and Transformers
- **Tong Hui Kang (huikang)** for the v27 warmstart adapter and the 15,979-problem algorithmic CoT corpus
  (`samvalladares/huikang-nemotron-artifacts`) used as the basis for v0.4 training [Rule 6]
- **kuangyicheng (AbsoluterGeist)** for publishing the 0.87/0.88 training approach
  ([kuangyicheng/nemotron-087-training](https://www.kaggle.com/code/kuangyicheng/nemotron-087-training))
  which the v0.5 approach replicates
- **Unsloth** for `FastLanguageModel` which enables joint MoE + attention LoRA training
